# hermes — Untreated HF, all 990 prompts

Select a **Colab GPU kernel** in VS Code. BF16 is fixed: A100/H100 recommended, L4 supported if memory permits. No automatic quantization, prompt truncation, or model substitution. A GPU change may alter timing/numerical behaviour; device details are recorded. Training checkpoints require identical software. Four adapters are loss-objective ablations over the same exact teacher response; masked fields remain visible. All four conditions are prespecified for validation and test, with no winner selection. Test remains a comparative replication.

## 1. Connect laptop receiver and restore code/data

Start the receiver, tunnel and watchdog using the commands in notebook 00, section 5. Paste the HTTPS tunnel URL and receiver token. This cell always prompts, including on rerun.

**Recovery / safe rerun:** Rerun after every runtime or tunnel restart. Do not replace code while an old worker is still running.

**Cost:** No paid API requests.



In [1]:
from pathlib import Path
from getpass import getpass
import json, io, os, sys, urllib.request, zipfile
ROOT = Path('/content/CAM_LDS_STREAMLINED')
ROOT.mkdir(parents=True, exist_ok=True)
receiver_url = input('Local receiver HTTPS tunnel URL: ').strip().rstrip('/')
receiver_token = getpass('Local receiver token: ').strip()
from urllib.parse import urlparse
if urlparse(receiver_url).scheme != 'https':
    raise ValueError('Enter the https:// URL printed by cloudflared')
request = urllib.request.Request(receiver_url+'/bundle', headers={'Authorization':'Bearer '+receiver_token})
with urllib.request.urlopen(request, timeout=120) as response:
    bundle = response.read()
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.infolist():
        target = (ROOT/member.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Invalid bundle path')
    archive.extractall(ROOT)
(ROOT/'runtime').mkdir(exist_ok=True)
connection = ROOT/'runtime/connection.json'
connection.write_text(json.dumps({'url':receiver_url,'token':receiver_token}), encoding='utf-8')
os.chmod(connection, 0o600)
del receiver_token, bundle
sys.path.insert(0, str(ROOT))
print('Bootstrap restored. No API/HF key is stored in notebook output.')

Bootstrap restored. No API/HF key is stored in notebook output.


## 2. Install reusable GPU dependencies

Install pinned CUDA PyTorch, Transformers and PEFT in the kernel environment before any model imports. **No llama.cpp/CUDA compilation is required.** Base models download directly from their frozen Hugging Face revisions and are reused within the surviving Colab runtime; large base-weight archives never pass through the laptop receiver.

**Recovery / safe rerun:** Rerun 1 then 2 on a new runtime. Rerunning on the same runtime reuses installed packages.

**Cost:** No paid API requests.

Colab time while installation/downloads run; no inference or training yet.

In [2]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-colab.txt')],check=True)
print('Dependencies ready; model work runs in a fresh subprocess.')

Dependencies ready; model work runs in a fresh subprocess.


## 3. Verify connection, frozen configuration and GPU

Requires completed preparation section 4. Secrets are read from the laptop secret folder by the authenticated receiver, never embedded into notebooks. Do not launch the same model/condition in overlapping notebooks.

**Recovery / safe rerun:** New Colab runtime: rerun 1–3, then the interrupted stage. Tunnel change: rerun 1 and 3, then that stage. For an HTTP/authentication error, use notebook 00 section 5 to restart receiver/tunnel, then rerun 1 and 3.

**Cost:** No paid API requests.



In [3]:
from streamlined.common import read_json
from streamlined.transport import Mirror
from streamlined.notebook import worker
connection = read_json(ROOT/'runtime/connection.json')
mirror = Mirror(connection['url'],connection['token'])
health = mirror.call('/health')
print(health)
if not health.get('watchdog_alive'):
    raise RuntimeError('Start the local watchdog using the commands above this cell')
manifest = read_json(ROOT/'manifests/experiment.json')
print('Experiment hash:',manifest['config_hash'])
MODEL = 'hermes'
import subprocess
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],text=True))

{'protocol': 'camlds-streamlined-v1', 'watchdog_alive': True}
Experiment hash: 49e13df7571913d2c39067dfd7dcde93353110b2f4c7523bc8a18de2459b4b73
NVIDIA A100-SXM4-80GB, 81920 MiB, 580.82.07



## 4. Generate all 990 untreated responses

The exact HF base revision, BF16 precision and template match the SFT pipeline. Greedy decoding with a 1,024-token output ceiling; only output allowance is reduced if the unchanged prompt leaves less context. Unsupported rows receive an explicit status. CSVs update on the laptop after each response.

**Recovery / safe rerun:** New Colab runtime: rerun 1–3, then the interrupted stage. Tunnel change: rerun 1 and 3, then that stage. Rerun 4. Saved completed rows are skipped, errors retried.

**Cost:** GPU inference for up to 990 rows.



In [4]:
worker(ROOT,MODEL,'infer','untreated_hf','all')

Started hermes/infer/untreated_hf/all; PID=51109; log=/content/CAM_LDS_STREAMLINED/telemetry/logs/hermes_infer_untreated_hf_all.log
I0000 00:00:1793027792.023272   51109 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793027792.098831   51109 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793027794.609084   51109 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_

## 7. Record Colab billing observations

Optional account balances supplement measured device telemetry. Concurrent sessions make account-balance attribution ambiguous; that scope is retained in the CSV.

**Recovery / safe rerun:** New Colab runtime: rerun 1–3, then the interrupted stage. Tunnel change: rerun 1 and 3, then that stage. Use the same session label to correct a record; do not infer balances from GPU utilization.

**Cost:** No paid API requests.



In [ ]:
from streamlined.notebook import billing
SESSION = input('Session label (unique per allocated runtime): ').strip()
START_UNITS = input('Balance before this session (blank if unknown): ').strip()
END_UNITS = input('Current balance (blank if unknown): ').strip()
if START_UNITS and END_UNITS:
    billing(ROOT,SESSION,float(START_UNITS),float(END_UNITS))
    from streamlined.common import digest
    mirror.put(ROOT,ROOT/'telemetry/billing'/f'{digest(SESSION)}.json')